# term

> A client for gateway-hosted terminals: REST lifecycle plus one websocket attachment

In [ ]:
#| default_exp term

`JupyAsyncTerminalClient` connects to terminals hosted by [rustygate](https://github.com/AnswerDotAI/rustygate). An app can use it to provide a shell on the same machine and filesystem as its kernels, including inside a container. The gateway uses [ptymini](https://github.com/AnswerDotAI/ptymini) to manage the terminals.

The client uses `KernelApi` for terminal lifecycle requests at `/api/terminals` and a websocket for input, output and terminal controls.

In [ ]:
#| export
import json, websockets
from contextlib import suppress
from fastcore.basics import patch
from jupyasyncclient.core import KernelApi, _join_url

In [ ]:
import asyncio, os, time
from fastcore.test import test_eq
from rustygate.tools import start_gateway


## Lifecycle

Pass a gateway URL when constructing the client. To use an existing terminal, also pass its `name`. Authentication, timeout and HTTP options are the same as for `JupyAsyncKernelClient`.

`start_terminal` creates a terminal and stores its name on the client. It passes the gateway's creation options through unchanged: `argv`, `cwd`, `env`, `appendenv`, `rc`, `rows`, `cols` and `username`.

In [ ]:
#| export
class JupyAsyncTerminalClient(KernelApi):
    "One gateway terminal: REST lifecycle through the spec ops on `api`, plus one ws attachment."
    def __init__(self, base_url, name=None, token=None, headers=None, timeout=30, http_client=None, verify=True):
        super().__init__(base_url, token=token, headers=headers, timeout=timeout, http_client=http_client, verify=verify)
        self.name,self._ws = name,None

    def _tpath(self, name=''): return f'/api/terminals/{name}' if name else '/api/terminals'
    async def list_terminals(self): return await self.api.terminals.list_terms()

    async def start_terminal(self, **kw):
        "Create a terminal (gateway creation options pass through) and bind its name."
        model = await self.api.terminals.create_term(**kw)
        self.name = model['name']
        return model

    async def shutdown_terminal(self): return await self.api.terminals.delete_term(name=self.name)

Start a gateway and a Bash shell without startup files. A fixed prompt makes the example output predictable.

In [ ]:
BASH = ['bash', '--norc', '--noprofile', '-i']
BENV = dict(os.environ, PS1='$ ', TERM='dumb')
g = start_gateway()
tc = JupyAsyncTerminalClient(g.url)
test_eq(await tc.list_terminals(), [])
model = await tc.start_terminal(argv=BASH, env=BENV)
test_eq((await tc.list_terminals())[0]['name'], tc.name)
model

```python
{'alive': True, 'last_activity': 1789266815.656605, 'name': '1', 'path': None}
```

## The channel

`connect` opens the websocket and returns the server's `setup` frame. The gateway then replays available scrollback as binary frames.

The websocket uses jupygate's protocol. Binary frames carry terminal bytes unchanged in either direction. Text frames contain JSON controls:

- The server sends `setup` when a client connects.
- `gap` reports that the client fell behind the server's replay buffer.
- `eof` reports the terminal's exit code.
- The client sends `set_size` to resize the terminal.

`write` sends bytes to the terminal. `resize` sends `set_size`. `frames` yields output as `bytes` and controls as dictionaries. Use one `frames` iterator per connection. Multiple iterators would compete to read the same socket.

`aclose` closes this client's websocket attachment. It does not delete the terminal. Use `shutdown_terminal` to stop the terminal for all clients.

In [ ]:
#| export
@patch
async def connect(self:JupyAsyncTerminalClient):
    "Open the terminal's ws channel and return the `setup` frame; replayed scrollback follows as binary."
    params = dict(token=self.token) if self.token else None
    url = _join_url(self.base_url, self._tpath(self.name)+'/channel', ws=True, params=params)
    self._ws = await websockets.connect(url, ssl=self._ws_ssl(url), max_size=None)
    return json.loads(await self._ws.recv())

@patch
async def write(self:JupyAsyncTerminalClient, data:bytes): await self._ws.send(data)

@patch
async def resize(self:JupyAsyncTerminalClient, rows:int, cols:int):
    await self._ws.send(json.dumps(dict(type='set_size', rows=rows, cols=cols)))

@patch
async def frames(self:JupyAsyncTerminalClient):
    "Incoming frames: pty output as `bytes`, control frames as parsed dicts. Ends when the ws closes."
    with suppress(websockets.ConnectionClosed):
        async for frame in self._ws: yield frame if isinstance(frame, bytes) else json.loads(frame)

@patch
async def aclose(self:JupyAsyncTerminalClient):
    if self._ws is not None:
        with suppress(Exception): await self._ws.close()
        self._ws = None

Send a command and read its output. Frame boundaries depend on timing, so `read_until` accumulates bytes until the expected text appears.

In [ ]:
async def read_until(fr, pat:bytes, timeout=10.0)->bytes:
    "Accumulate binary frames from async iterator `fr` until `pat` appears (control frames are skipped)."
    buf = b''
    end = time.monotonic() + timeout
    while pat not in buf:
        item = await asyncio.wait_for(anext(fr), end - time.monotonic())
        if isinstance(item, bytes): buf += item
    return buf

setup = await tc.connect()
test_eq(setup['type'], 'setup')
fr = tc.frames()
await tc.write(b'echo wired $((6*7))\n')
out = await read_until(fr, b'wired 42')
out[-20:]

b'$((6*7))\r\nwired 42\r\n'

A second client can connect to the same terminal name and read its scrollback. This lets an app recover earlier output after a restart or browser refresh.

In [ ]:
tc2 = JupyAsyncTerminalClient(g.url, name=tc.name)
test_eq((await tc2.connect())['type'], 'setup')
fr2 = tc2.frames()
replay = await read_until(fr2, b'wired 42')
b'echo wired' in replay, b'wired 42' in replay

(True, True)

Check the resized terminal with `stty size`. Either client can shut it down by name. All attached clients then receive `eof` with the terminal's exit code.

In [ ]:
await tc.resize(50, 120)
await tc.write(b'stty size\n')
assert b'50 120' in await read_until(fr, b'50 120')
await tc2.shutdown_terminal()
item = await asyncio.wait_for(anext(fr), 10)
while isinstance(item, bytes): item = await asyncio.wait_for(anext(fr), 10)  # drain final output; the dict is the eof
test_eq(item['type'], 'eof')
test_eq(await tc.list_terminals(), [])
item

{'type': 'eof', 'code': -1}

In [ ]:
#| hide
await tc.aclose()
await tc2.aclose()
g.stop()

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()